# DREAM NOTEBOOK — Kaggle GPU connector

This is a small **compute connector only**. It is not where you edit or run DREAM training cells.

1. Enable a Kaggle GPU and Internet.
2. Add Kaggle Secrets `DREAM_HOST_NOTEBOOK_WORKER_TOKEN` and `DREAM_NOTEBOOK_ARTIFACT_TOKEN`. Add `HF_TOKEN` only if you also plan to improve the existing Qwen adapter.
3. Run the single code cell below and leave it running.
4. Open DREAM NOTEBOOK in another tab. The native and adapter workflows run there; this Kaggle notebook only supplies GPU compute.

Do not run the separate Qwen chat worker in the same GPU session while either training workflow is running; it would compete for GPU memory.

In [ ]:
# DREAM NOTEBOOK GPU connector
# This notebook only connects Kaggle compute to DREAM NOTEBOOK.
# Actual cells, experiments and training workflow live at:
# https://dream-host-live.onrender.com/notebook

import os
import sys
import hashlib
import subprocess
from pathlib import Path

import requests
import torch
from kaggle_secrets import UserSecretsClient

HOST = "https://dream-host-live.onrender.com"
WORKER_SOURCE = (
    "https://raw.githubusercontent.com/mjnr0477/dream-ai/"
    "dream-host-runtime/dream-host/notebook_worker.py"
)

if not torch.cuda.is_available():
    raise RuntimeError("Turn on a Kaggle GPU accelerator before starting the DREAM NOTEBOOK worker.")

secrets = UserSecretsClient()

def read_secret(name):
    try:
        value = str(secrets.get_secret(name) or "").strip()
    except Exception as exc:
        raise RuntimeError("Missing Kaggle Secret: " + name + ". Add it in Add-ons → Secrets.") from exc
    if len(value) < 32:
        raise RuntimeError("Kaggle Secret " + name + " is missing or too short.")
    return value

# This is a distinct owner-configured token, NOT DREAM_HOST_WORKER_TOKEN.
os.environ["DREAM_HOST_URL"] = HOST
os.environ["DREAM_HOST_NOTEBOOK_WORKER_TOKEN"] = read_secret("DREAM_HOST_NOTEBOOK_WORKER_TOKEN")
os.environ["DREAM_NOTEBOOK_ARTIFACT_TOKEN"] = read_secret("DREAM_NOTEBOOK_ARTIFACT_TOKEN")
# HF_TOKEN is needed only for the optional Qwen adapter-improvement workflow.
# Native DREAM-from-scratch training does not require Hugging Face credentials.
try:
    os.environ["DREAM_NOTEBOOK_HF_TOKEN"] = read_secret("HF_TOKEN")
    print("Optional Hugging Face credential: available for adapter improvement.")
except RuntimeError:
    os.environ.pop("DREAM_NOTEBOOK_HF_TOKEN", None)
    print("HF_TOKEN not set; native DREAM training remains available.")
os.environ["DREAM_HOST_NOTEBOOK_TRUSTED_MODE"] = "1"

print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)
print("DREAM HOST:", HOST)
print("Downloading the first-party DREAM NOTEBOOK worker…")

response = requests.get(WORKER_SOURCE, timeout=60)
response.raise_for_status()
worker_file = Path("/kaggle/working/dream_notebook_worker.py")
worker_file.write_bytes(response.content)
print("Worker source SHA-256:", hashlib.sha256(response.content).hexdigest())
print("Starting the worker. Keep this Kaggle cell running while using DREAM NOTEBOOK.")
print("Open https://dream-host-live.onrender.com/notebook in another browser tab.")

# The worker executes notebook cells in a persistent Python kernel on this Kaggle GPU.
# It does not load the Qwen chat model. Stop this cell to disconnect compute.
subprocess.run([sys.executable, "-u", str(worker_file)], check=True)
